In [13]:
# Loading the data as in HW02:

from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [14]:
"""
Generating ground truth
"""

PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"
!wget $PREFIX/01-agentic-rag/code/rag_helper.py
!wget $PREFIX/04-evaluation/code/evaluation_utils.py

--2026-07-13 15:48:46--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py.3’

rag_helper.py.3     100%[===================>]   2.08K  --.-KB/s    in 0.001s  

2026-07-13 15:48:46 (1.76 MB/s) - ‘rag_helper.py.3’ saved [2134/2134]

--2026-07-13 15:48:47--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/code/evaluation_utils.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting respo

In [15]:
# The module's instructions generate questions from a FAQ record, so we adapt them for a lesson page:

data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [8]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [16]:
"""
Q1. Generating questions:

Generating questions for all 72 pages costs money and takes time, so let's start small and generate questions for just the first 3 pages:

01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/02-environment.md
01-agentic-rag/lessons/03-rag.md
Each call returns the token usage, which most LLM APIs report on the response object (e.g. response.usage.input_tokens / prompt_tokens).

What's the average number of input tokens across these 3 calls?

"""

# generating questions only for LLM Zoomcamp course:

documents_llm = []

for doc in documents:
    if doc["filename"] == "01-agentic-rag/lessons/01-intro.md":
        documents_llm.append(doc)
    elif doc["filename"] == "01-agentic-rag/lessons/02-environment.md":
        documents_llm.append(doc)
    elif doc["filename"] == "01-agentic-rag/lessons/03-rag.md":
        documents_llm.append(doc)

len(documents_llm)

3

In [17]:
# importing model

from dotenv import load_dotenv
from openai import OpenAI
import json

load_dotenv()
openai_client = OpenAI()

In [25]:
"""
Generating questions with structured output
We want the output as a list of strings, so we define that structure with a Pydantic model:
"""

from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [26]:
input_tokens_list = []

for doc in documents_llm:
    
    # preparing the document as json:
    user_prompt = json.dumps(doc)

    # creating the msg:
    messages = [
        {"role": "developer", "content": data_gen_instructions},
        {"role": "user", "content": user_prompt}]

    # calling the model:
    response = openai_client.responses.parse(
        model="gpt-5.4-mini",
        input=messages,
        text_format=Questions
    )
    
    tokens = response.usage.input_tokens
    input_tokens_list.append(tokens)
    print(f"Input tokens: {tokens}")

avg = sum(input_tokens_list) / len(input_tokens_list)
print(f"\nAverage input tokens: {avg}")

Input tokens: 1020
Input tokens: 1286
Input tokens: 1753

Average input tokens: 1353.0


In [29]:
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"
!wget $PREFIX/cohorts/2026/04-evaluation/ground-truth.csv

--2026-07-13 12:59:22--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/04-evaluation/ground-truth.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 48627 (47K) [text/plain]
Saving to: ‘ground-truth.csv’

ground-truth.csv    100%[===================>]  47.49K  --.-KB/s    in 0.02s   

2026-07-13 12:59:22 (2.46 MB/s) - ‘ground-truth.csv’ saved [48627/48627]



In [25]:
# using the dataset already created:

import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [27]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

In [34]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [35]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [23]:
q = ground_truth[0]["question"]
print("Question:", q)

NameError: name 'ground_truth' is not defined

In [42]:
!uv add minsearch

Resolved 170 packages in 10ms
Checked 86 packages in 128ms


In [29]:
import minsearch
from minsearch import Index

# Build text index
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(chunks)

# Define text_search
def text_search(query, num_results=5):
    return index.search(query, num_results=num_results)

# Run text search
# Q2
q = ground_truth[0]["question"]
print("Question:", q)

results = text_search(q)
print("First result filename:", results[0]["filename"])

Question: What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?
First result filename: 01-agentic-rag/lessons/03-rag.md


In [30]:
import pandas as pd

gt_url = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/04-evaluation/ground-truth.csv"
ground_truth = pd.read_csv(gt_url).to_dict(orient="records")

print(ground_truth[0])

q = ground_truth[0]["question"]
results = text_search(q)
print("First result filename:", results[0]["filename"])

{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?", 'filename': '01-agentic-rag/lessons/01-intro.md'}
First result filename: 01-agentic-rag/lessons/03-rag.md


In [57]:
# Download embedder if needed
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed"
!wget $PREFIX/embedder.py

--2026-07-13 15:38:42--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/embedder.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1520 (1.5K) [text/plain]
Saving to: ‘embedder.py’

embedder.py         100%[===================>]   1.48K  --.-KB/s    in 0.001s  

2026-07-13 15:38:43 (1.17 MB/s) - ‘embedder.py’ saved [1520/1520]



In [1]:
!pip install onnxruntime tokenizers --break-system-packages

Defaulting to user installation because normal site-packages is not writeable
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 865.4 kB/s eta 0:00:00 0:00:01
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 15.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 15.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 14.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 14.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/2

In [4]:
import sentence_transformers
print(sentence_transformers.__version__) 
# 0.2.5.1 is outdated -->

0.2.5.1


In [6]:
import urllib.request
from pathlib import Path

Path("models/Xenova/all-MiniLM-L6-v2/onnx").mkdir(parents=True, exist_ok=True)

base = "https://huggingface.co/Xenova/all-MiniLM-L6-v2/resolve/main/"

for f in ["tokenizer.json", "onnx/model.onnx"]:
    dest = Path("models/Xenova/all-MiniLM-L6-v2") / f
    dest.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {f}...")
    urllib.request.urlretrieve(base + f, dest)

print("Done!")

Done!


In [9]:
import urllib.request
from pathlib import Path

Path("models/Xenova/all-MiniLM-L6-v2").mkdir(parents=True, exist_ok=True)

base = "https://huggingface.co/Xenova/all-MiniLM-L6-v2/resolve/main/"

# tokenizer.json in root, model.onnx directly in model folder (not onnx/)
files = {
    "tokenizer.json": "models/Xenova/all-MiniLM-L6-v2/tokenizer.json",
    "onnx/model.onnx": "models/Xenova/all-MiniLM-L6-v2/model.onnx",
}

for src, dest in files.items():
    print(f"Downloading {src} -> {dest}...")
    urllib.request.urlretrieve(base + src, dest)

print("Done!")

Done!


In [10]:
from embedder import Embedder
embedder = Embedder()
print("Embedder loaded!")

Embedder loaded!


In [31]:
"""
Q3. First result with vector search
After running vector_search for the same question, what's the filename of the first result?

01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/03-rag.md
04-evaluation/lessons/11-evaluation-intro.md
04-evaluation/lessons/12-rag-answers.md
"""


vector_index = VectorSearch(keyword_fields=["filename"])
vector_index.fit(X, chunks)

def vector_search(query, num_results=5):
    v = embedder.encode(query)
    return vector_index.search(v, num_results=num_results)

results = vector_search(q)
print("First result filename:", results[0]["filename"])

First result filename: 01-agentic-rag/lessons/01-intro.md


In [32]:
"""
Q4. Evaluating text search
Evaluate text_search on the ground truth data.

What's the Hit Rate?

0.55
0.66
0.76
0.88
"""

def compute_relevance(question, search_fn, num_results=5):
    results = search_fn(question["question"], num_results=num_results)
    relevance = [r["filename"] == question["filename"] for r in results]
    return relevance

def hit_rate(relevance_list):
    return sum(any(r) for r in relevance_list) / len(relevance_list)

def mrr(relevance_list):
    scores = []
    for relevance in relevance_list:
        for i, r in enumerate(relevance):
            if r:
                scores.append(1 / (i + 1))
                break
        else:
            scores.append(0)
    return sum(scores) / len(scores)

def evaluate(ground_truth, search_fn):
    relevance_list = [compute_relevance(q, search_fn) for q in ground_truth]
    return {
        "hit_rate": hit_rate(relevance_list),
        "mrr": mrr(relevance_list)
    }

# Evaluate text search
results = evaluate(ground_truth, text_search)
print(results)

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592593}


In [33]:
"""
Q5. Evaluating vector search
Now evaluate vector_search - the part we left for the homework, since the module only evaluated keyword search.

What's the MRR?

0.35
0.45
0.55
0.65
"""

results = evaluate(ground_truth, vector_search)
print(results)

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}


In [34]:
"""
Q6. Tuning hybrid search
The k constant in RRF controls how much the top ranks matter. A smaller k sharpens the gap between positions, so being at the top of a list counts for more. The RRF paper uses 60 as a default, but the best value depends on the data

so let's measure it.
Evaluate hybrid_search over the full ground truth dataset for k values 1, 50, 100, and 200. Compare the MRR values for these runs.

Which k gives the best MRR?

1
50
100
200
Several values of k may give the same MRR. If there's a tie, pick the smallest k.
"""

def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60, num_results=5):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

# Evaluate for different k values
for k in [1, 50, 100, 200]:
    search_fn = lambda q, num_results=5, k=k: hybrid_search(q, k=k, num_results=num_results)
    results = evaluate(ground_truth, search_fn)
    print(f"k={k}: MRR={results['mrr']:.4f}, Hit Rate={results['hit_rate']:.4f}")

k=1: MRR=0.6482, Hit Rate=0.8389
k=50: MRR=0.6379, Hit Rate=0.8361
k=100: MRR=0.6379, Hit Rate=0.8361
k=200: MRR=0.6379, Hit Rate=0.8361


In [ ]:
# answer:
# higher MRR is better. MRR measures how high the correct result appears in the ranking: k=1